# 01 — Exploración de la base SEPA

**Qué es SEPA:** el Sistema Electrónico de Publicidad de Precios Argentinos. Los
supermercados están obligados por ley a publicar sus precios todos los días.
Son unos 12 a 14 millones de registros diarios.

**Objetivo de esta notebook:** entender qué hay adentro *antes* de calcular nada.

Preguntas que tengo que poder responder al final:

1. ¿Cuántas cadenas, sucursales y productos hay realmente por día, y es estable?
2. ¿Alguna cadena domina la muestra hasta el punto de sesgar un promedio?
3. ¿La cobertura geográfica permite comparar entre provincias?
4. ¿Qué forma tiene la distribución de precios?
5. ¿Qué proporción de la base entra en mi canasta de análisis?

In [ ]:
import os, sys, warnings
sys.path.insert(0, "..")
# Los datos crudos viven fuera del repo (pesan GB). Si moviste la carpeta,
# cambiá esta ruta o exportá RADAR_DATA_DIR antes de abrir el notebook.
os.environ.setdefault("RADAR_DATA_DIR", os.path.expanduser("~/sepa-data"))
warnings.filterwarnings("ignore")

import duckdb, pandas as pd, matplotlib.pyplot as plt
from src import config as cfg

pd.set_option("display.max_columns", 40); pd.set_option("display.width", 150)
plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold"})

PARQUET = str(cfg.INTERIM / "fecha=*" / "*.parquet")
con = duckdb.connect()
dias = sorted(p.name.split("=")[1] for p in cfg.INTERIM.glob("fecha=*"))
assert dias, f"No hay datos en {cfg.INTERIM}. Corré primero: python -m src.clean"
print(f"Datos: {cfg.DATA}")
print(f"{len(dias)} días — de {dias[0]} a {dias[-1]}")

## 1. Volumen y cobertura por día

Primer control: que el volumen sea estable. Un día con la mitad de las filas es
un día con un problema de origen, no un día en que bajaron los precios.

In [ ]:
vol = con.execute(f"""
    SELECT fecha,
           count(*)                                          AS filas,
           count(DISTINCT ean)                               AS eans,
           count(DISTINCT cadena)                            AS cadenas,
           count(DISTINCT id_comercio || '-' || id_sucursal) AS sucursales,
           count(DISTINCT provincia)                         AS provincias
    FROM read_parquet('{PARQUET}')
    GROUP BY 1 ORDER BY 1
""").df()
display(vol)

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
vol.plot(x="fecha", y="filas", marker="o", ax=ax[0], legend=False, color="#1f4e79")
ax[0].set_title("Filas por día"); ax[0].set_ylabel("registros")
vol.plot(x="fecha", y=["cadenas", "provincias"], marker="o", ax=ax[1])
ax[1].set_title("Cadenas y provincias que reportan"); ax[1].set_ylabel("cantidad")
plt.tight_layout()

> **Anotá acá lo que ves.** Si el número de cadenas cambia entre días, eso ya es
> un hallazgo: cualquier comparación temporal puede estar reflejando composición
> de la muestra en vez de un cambio real de precios.

## 2. Concentración: ¿quién domina la base?

Si una sola cadena aporta el 40% de las filas, cualquier promedio simple es en
realidad el precio de esa cadena. Este número es el que justifica usar **mediana
por cadena** en vez de promedio sobre la base cruda.

In [ ]:
conc = con.execute(f"""
    SELECT cadena,
           count(*)                                        AS filas,
           round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct,
           count(DISTINCT id_sucursal)                     AS sucursales,
           count(DISTINCT provincia)                       AS provincias,
           count(DISTINCT ean)                             AS eans
    FROM read_parquet('{PARQUET}')
    GROUP BY 1 ORDER BY filas DESC
""").df()
display(conc)

print(f"Las 3 cadenas más grandes concentran el {conc.pct.head(3).sum():.1f}% de los registros")
ax = conc.set_index("cadena")["pct"].head(12).iloc[::-1].plot.barh(color="#1f4e79")
ax.set_xlabel("% de los registros"); ax.set_title("Concentración por cadena");

## 3. Cobertura geográfica

Regla que me impongo: **solo comparo precios en provincias con al menos 3 cadenas**.
Con menos, la "cadena más cara" y la "más barata" pueden ser la misma dos veces.

In [ ]:
prov = con.execute(f"""
    SELECT provincia, count(*) AS filas,
           count(DISTINCT cadena) AS cadenas,
           count(DISTINCT id_comercio || '-' || id_sucursal) AS sucursales
    FROM read_parquet('{PARQUET}')
    GROUP BY 1 ORDER BY filas DESC
""").df()
display(prov)

comparables = (prov.cadenas >= 3).sum()
print(f"Provincias comparables (>=3 cadenas): {comparables} de {len(prov)}")
print(f"Cubren el {100*prov[prov.cadenas>=3].filas.sum()/prov.filas.sum():.1f}% de los registros")

## 4. Distribución de precios

En escala lineal no se ve nada: conviven el sachet de leche y el televisor. En
escala logarítmica aparece la estructura real, y suele verse más de una moda
(los distintos órdenes de magnitud de producto).

In [ ]:
px = con.execute(f"SELECT precio, precio_unitario FROM read_parquet('{PARQUET}') USING SAMPLE 300000 ROWS").df()

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
px.precio.plot.hist(bins=100, ax=ax[0], color="#1f4e79")
ax[0].set_title("Precio de lista (lineal)"); ax[0].set_xlabel("$")
px.precio.plot.hist(bins=100, log=True, ax=ax[1], color="#1f4e79")
ax[1].set_title("Precio de lista (eje Y logarítmico)"); ax[1].set_xlabel("$")
plt.tight_layout()

display(px.describe(percentiles=[.01, .25, .5, .75, .95, .99]).round(1))

## 5. El catálogo de productos

Acá aparece el problema que define el resto del proyecto: **el mismo EAN tiene
descripciones distintas en cada cadena**. Por eso la canasta se arma con patrones
de texto y no con códigos de barra.

In [ ]:
desc_por_ean = con.execute(f"""
    SELECT n_desc, count(*) AS eans FROM (
        SELECT ean, count(DISTINCT descripcion) AS n_desc
        FROM read_parquet('{PARQUET}') GROUP BY 1
    ) GROUP BY 1 ORDER BY 1
""").df()
display(desc_por_ean.head(12))

ejemplos = con.execute(f"""
    SELECT ean, count(DISTINCT descripcion) AS variantes,
           string_agg(DISTINCT descripcion, '  |  ') AS textos
    FROM read_parquet('{PARQUET}')
    GROUP BY 1 HAVING count(DISTINCT descripcion) BETWEEN 3 AND 6
    ORDER BY random() LIMIT 5
""").df()
for _, r in ejemplos.iterrows():
    print(f"\nEAN {r.ean} ({r.variantes} variantes):")
    for t in r.textos.split("  |  "):
        print("   ", t[:90])

## 6. ¿Cuánto de la base entra en la canasta?

`cobertura()` muestra, por item: cuántas filas capta, cuántos EAN distintos y
qué porcentaje viene en la unidad esperada.

Qué mirar:

- **un item con 2 o 3 EANs** → el patrón es demasiado estrecho, se pierde señal
- **un item con miles de EANs** → probablemente está mezclando productos distintos
- **`pct_unidad_ok` bajo** → el patrón trae otra presentación (por ejemplo,
  gaseosa vendida por unidad cuando esperabas litros)

In [ ]:
from src.canasta import cobertura

muestra = con.execute(f"""
    SELECT ean, descripcion, precio, precio_unitario, unidad_base
    FROM read_parquet('{PARQUET}') USING SAMPLE 500000 ROWS
""").df()

cob = cobertura(muestra)
display(cob)
print(f"\nLa canasta capta el {100*cob.filas.sum()/len(muestra):.1f}% de las filas de la muestra")

### Items para revisar

Los que tienen `pct_unidad_ok` bajo o muy pocos EANs son los que hay que corregir
en `src/canasta.py`. Este bloque los saca ordenados por prioridad.

In [ ]:
revisar = cob[(cob.get("pct_unidad_ok", 100) < 80) | (cob.eans < 5)]
if len(revisar):
    display(revisar.sort_values("filas", ascending=False))
else:
    print("Ningún item con problemas evidentes de cobertura.")

## Conclusiones

> Completá con lo que encontraste. Ejemplo de la forma que tiene que tener:
>
> - **N cadenas y M sucursales**, con variación entre días de X a Y cadenas.
>   → cualquier comparación temporal tiene que controlar por composición.
> - **La cadena Z concentra el W%** de los registros → uso mediana por cadena,
>   nunca promedio sobre la base cruda.
> - **K provincias tienen 3 o más cadenas** y cubren el V% de los datos → el
>   análisis de dispersión se limita a esas.
> - **El mismo EAN aparece con hasta N descripciones distintas** → los productos
>   se identifican por patrones de texto, no por código de barras.
> - **La canasta cubre el X%** de las filas, con todos los items por encima de
>   N EANs distintos.